# Tema 11 · Bloque 1 (Lunes) — Simulaciones cuánticas en ciencia de materiales
### Los 4 retos resueltos · 100% local, sin token de IBM Quantum

**Cómo usar este notebook:** ejecuta de arriba hacia abajo. Cada reto trae:
1. **"Entiende esto primero"** — los conceptos en lenguaje llano.
2. El **código** que produce el resultado.
3. La **respuesta escrita** a la pregunta del reto.

---

## La pregunta que responde TODO el bloque 1

> **¿Por qué no simulamos las moléculas con las computadoras que ya tenemos?**

La respuesta corta: **no es un problema de velocidad, es de tamaño**. El problema crece tan rápido que ninguna
computadora clásica alcanza, por grande que sea. A eso se le llama **la pared algorítmica**.

## Mapa mental en 6 líneas

| Concepto | En cristiano |
|---|---|
| **Simulación computacional** | Predecir cómo se comporta un material **antes** de fabricarlo. |
| **Ecuación de Schrödinger** | La ecuación que describe cómo evoluciona un sistema cuántico. Resolverla = conocer la molécula. |
| **Pared algorítmica** | Al resolverla para muchos electrones, la memoria necesaria crece como **2ⁿ**. Te estrellas. |
| **Correlación electrónica** | Los electrones se esquivan entre sí. Modelar eso clásicamente cuesta exponencialmente. |
| **Qubit** | En vez de 0 **o** 1, una nube de probabilidades — que es justo como se comporta un electrón. |
| **Backend** | El procesador donde corres el circuito: un **simulador** o una **QPU** real. |

> **La frase del bloque:** *cada electrón que añades no suma dificultad, la **multiplica**.*

---

## 0. Preparación del entorno

Solo se necesitan `qiskit`, `numpy` y `matplotlib`. **No hace falta token de IBM**: todo lo de este notebook
corre en tu propia máquina.

> 📌 **En Google Colab** `qiskit` no viene preinstalado. La celda de abajo lo detecta y lo instala sola
> (~30 s la primera vez). Si tras instalar sigue fallando el import: *Entorno de ejecución → Reiniciar sesión*.


In [ ]:
try:
    import qiskit
except ModuleNotFoundError:
    print("qiskit no encontrado -> instalando...")
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "qiskit"], check=True)
    import qiskit

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit.providers.basic_provider import BasicSimulator

print("Entorno listo · qiskit", qiskit.__version__)
print("Todo corre en local, sin credenciales ni token de IBM.")


---
# RETO 1 — Conexión inicial y selección de backends

## 🧠 Entiende esto primero

### ¿Qué es un "backend"?
Es simplemente **la máquina donde se ejecuta tu circuito**. Tú escribes el circuito una sola vez; luego eliges
dónde corre. Hay dos familias, y confundirlas es el error clásico:

| | **Simulador** | **QPU (procesador cuántico real)** |
|---|---|---|
| Qué es | Un programa que **imita** la física cuántica con matemáticas clásicas | Hardware físico: qubits superconductores a ~15 milikelvin |
| Ruido | **Ninguno** (o el que tú decidas añadir) | Sí: decoherencia, errores de compuerta y de lectura |
| Resultado | El valor **exacto**, ideal | Un valor con desviación, hay que mitigarlo |
| Límite | ~30-50 qubits: **choca con la pared algorítmica** | 127+ qubits, pero con ruido |
| Espera | Instantánea | Cola compartida: minutos u horas |
| Costo | Gratis, en tu laptop | Tiempo de QPU (se cobra o se raciona) |

### ¿Qué es `QiskitRuntimeService`?
Es la **puerta de entrada a la nube de IBM**. Al crearlo, se autentica con tu token y te deja pedirle la lista de
máquinas disponibles. Sin token no puedes usarlo — y por eso este notebook trabaja con un **simulador local**,
que es idéntico conceptualmente y no necesita credenciales.

### ¿Por qué el token no va en el código?
Un token es una **credencial personal**, equivalente a una contraseña. Si lo escribes en el notebook y lo subes a
GitHub o lo compartes con un compañero, cualquiera puede consumir tu tiempo de cómputo. Por eso se guarda una vez
en la configuración local (o en una variable de entorno) y el código solo dice `QiskitRuntimeService()`.

### La regla práctica del flujo de trabajo
> **Desarrollas y depuras en simulador. Solo cuando el circuito ya funciona, lo mandas a la QPU.**

Mandar a la QPU un circuito con un bug es tirar tiempo de cola y de presupuesto a la basura.


## 1.1 El código del reto (versión nube — requiere token)

Esta es la versión del PDF. La dejo **comentada** para que no falle: solo funciona con un token de IBM configurado.


In [ ]:
# --- VERSIÓN NUBE (requiere token de IBM Quantum) ---
# Descomenta solo si ya guardaste tu token. Instalación: pip install qiskit-ibm-runtime
#
# from qiskit_ibm_runtime import QiskitRuntimeService
#
# # Guardar el token UNA sola vez (nunca dejes el token escrito en un notebook que compartes):
# # QiskitRuntimeService.save_account(channel="ibm_quantum_platform", token="TU_TOKEN")
#
# service = QiskitRuntimeService()          # lee el token guardado
# for backend in service.backends():
#     print(f"Backend: {backend.name} | Qubits: {backend.num_qubits}")

print("Salida típica en la nube de IBM (para referencia):\n")
backends_ejemplo = [
    ("ibm_brisbane",        127, "QPU real",  "Eagle r3"),
    ("ibm_kyiv",            127, "QPU real",  "Eagle r3"),
    ("ibm_sherbrooke",      127, "QPU real",  "Eagle r3"),
    ("ibmq_qasm_simulator",  32, "Simulador", "sin ruido"),
]
print(f"{'Backend':<24}{'Qubits':>8}   {'Tipo':<12}{'Nota'}")
print("-" * 62)
for nombre, nq, tipo, nota in backends_ejemplo:
    print(f"{nombre:<24}{nq:>8}   {tipo:<12}{nota}")


## 1.2 Versión local: un backend de verdad, sin token

`BasicSimulator` es un backend real de Qiskit que vive en tu máquina. La **interfaz es la misma** que la de una
QPU: le pasas un circuito, le pides `shots` (repeticiones) y te devuelve resultados. Cambiar de simulador a QPU
es cambiar una sola línea.


In [ ]:
simulador = BasicSimulator()

print("BACKEND LOCAL DISPONIBLE")
print("-" * 45)
print(f"Nombre       : {simulador.name}")
nq = simulador.num_qubits
print(f"Qubits máx.  : {nq if nq else 'sin tope fijo — lo limita tu RAM (~30 qubits en la práctica)'}")
print(f"Tipo         : simulador ideal (sin ruido)")
print(f"Requiere red : NO")
print(f"Requiere token: NO")

# Un circuito mínimo para comprobar que el backend responde
qc = QuantumCircuit(2)
qc.h(0)          # superposición en el qubit 0
qc.cx(0, 1)      # entrelaza el qubit 0 con el 1
qc.measure_all()

print("\nCircuito de prueba (estado de Bell):")
print(qc.draw(output="text"))

resultado = simulador.run(transpile(qc, simulador), shots=1024).result()
print("\nResultados de 1024 mediciones:", resultado.get_counts())
print("\nSolo salen '00' y '11', nunca '01' ni '10': los dos qubits están ENTRELAZADOS.")
print("En una QPU real verías además un pequeño porcentaje de '01' y '10' -> eso es el RUIDO.")


## 1.3 Simulador vs. QPU: cuándo usar cada uno

In [ ]:
decisiones = [
    ("Estoy escribiendo el circuito y tiene bugs",        "SIMULADOR", "Iteración instantánea y gratis"),
    ("Quiero el valor exacto de referencia (sin ruido)",  "SIMULADOR", "La QPU nunca te da el valor ideal"),
    ("Mi sistema tiene menos de ~30 qubits",              "SIMULADOR", "Todavía cabe en memoria clásica"),
    ("Necesito validar el comportamiento del hardware",   "QPU",       "El ruido real no se puede fingir bien"),
    ("Mi sistema pasa de ~50 qubits",                     "QPU",       "El simulador ya chocó con la pared algorítmica"),
    ("Voy a publicar/certificar un resultado",            "QPU",       "Se exige ejecución en hardware real"),
]

print(f"{'Situación':<52}{'Dónde':<12}Por qué")
print("-" * 110)
for situacion, donde, razon in decisiones:
    print(f"{situacion:<52}{donde:<12}{razon}")


## ✅ Respuesta al Reto 1

**Pregunta:** *¿por qué es indispensable distinguir entre un backend basado en simulador local y un procesador
cuántico real (QPU) al planificar experimentos de química computacional?*

**Respuesta:**

Porque **responden preguntas distintas** y cuestan recursos distintos. Confundirlos invalida el experimento o
desperdicia presupuesto. Hay cinco diferencias que cambian la planificación:

1. **Ruido: el simulador da el valor ideal, la QPU el valor real.** El simulador resuelve la física exactamente,
   así que te da la energía correcta. La QPU tiene decoherencia y errores de compuerta, así que devuelve un valor
   desviado. En química eso importa muchísimo: la **precisión química** exigida es ~1 kcal/mol, y el ruido actual
   la supera con facilidad. Sin un valor de referencia del simulador **no puedes saber cuánto se equivocó la QPU**.

2. **Tamaño: el simulador tiene un techo duro.** Simular clásicamente requiere guardar 2ⁿ amplitudes, así que
   alrededor de 30-50 qubits el simulador **choca con la misma pared algorítmica** que queremos superar. Si tu
   molécula cabe ahí, no hay razón para usar hardware cuántico; si no cabe, el simulador deja de ser opción.

3. **Costo y cola: los recursos son opuestos.** El simulador es gratis e instantáneo. La QPU es un recurso
   compartido con cola y tiempo tarifado. Depurar en QPU es tirar dinero y horas.

4. **Metodología: se necesitan ambos, en orden.** El flujo profesional es *desarrollar y validar en simulador →
   ejecutar en QPU → comparar ambos resultados*. Esa comparación **es** la medida del error del hardware y lo que
   justifica aplicar mitigación (ZNE, PEA).

5. **Conectividad y transpilación.** Una QPU real tiene una topología física concreta: no todos los qubits están
   conectados entre sí. El circuito debe **transpilarse** a esa topología, lo que añade compuertas SWAP y por tanto
   más ruido. El simulador ideal no tiene esa restricción, así que un circuito que corre perfecto en simulador
   puede volverse mucho más profundo y ruidoso en hardware.

**En una frase:** el simulador te dice **cuál es la respuesta correcta**; la QPU te dice **qué tan cerca está hoy
el hardware de poder dártela**. Planificar sin distinguirlos lleva a confundir un error de programación con un
error de hardware.


---
# RETO 2 — La pared algorítmica y el crecimiento exponencial

## 🧠 Entiende esto primero

### De dónde sale el 2ⁿ
Cada electrón (más precisamente, cada **espín-orbital**) puede estar **ocupado o vacío**: 2 opciones. Con `n`
orbitales, el número de configuraciones posibles es `2 × 2 × … × 2` = **2ⁿ**.

Y aquí está el detalle que lo cambia todo: por la superposición, la molécula **no está en una configuración**,
sino en una mezcla de **todas** a la vez. Para describirla exactamente hay que guardar **un número (una amplitud)
por cada configuración**. O sea 2ⁿ números.

### Qué significa "exponencial" en la práctica
No es "crece rápido". Es que **cada electrón que añades DUPLICA** la memoria necesaria:

```
20 electrones →      1,048,576 amplitudes
21 electrones →      2,097,152     (el doble)
22 electrones →      4,194,304     (el doble otra vez)
```

Añadir 10 electrones no suma 10 veces el trabajo: lo multiplica por **1024**.

### Por qué "más RAM" no es la solución
El hardware clásico mejora **linealmente** (o como mucho duplicando cada par de años). El problema crece
**exponencialmente con cada electrón**. Es una carrera perdida por construcción: toda esa inversión en memoria
te compra **un electrón más**. Luego otra vez lo mismo.


In [ ]:
# El cálculo que pide el reto: n = 20 electrones activos
n = 20
estados = 2**n
bytes_por_amplitud = 16          # un número complejo en doble precisión

print(f"Molécula con n = {n} electrones activos")
print("-" * 52)
print(f"Espacio de estados (2^n)  : {estados:,} configuraciones")
print(f"Memoria (16 bytes c/u)    : {estados*bytes_por_amplitud/1024**2:,.1f} MB")

print(f"\nY ahora añadimos UN electrón más:")
print(f"n = {n+1}  ->  {2**(n+1):>13,} estados   ({2**(n+1)/estados:.0f}x mas que con {n})")
print(f"n = {n+2}  ->  {2**(n+2):>13,} estados   ({2**(n+2)/estados:.0f}x mas que con {n})")
print(f"n = {n+10}  ->  {2**(n+10):>13,} estados   ({2**(n+10)/estados:,.0f}x mas que con {n})")
print("\nCada electrón DUPLICA la exigencia. No suma: multiplica.")


In [ ]:
# ¿Hasta dónde llega cada máquina?
GB = 1024**3
maquinas = [
    ("Laptop (16 GB)",            16 * GB),
    ("Workstation (128 GB)",     128 * GB),
    ("Nodo HPC (2 TB)",         2048 * GB),
    ("Frontier (~9.2 PB)",   9_200_000 * GB),
]

print(f"{'Máquina':<26}{'RAM':>16}{'Máx. electrones':>18}")
print("-" * 62)
for nombre, ram in maquinas:
    k = 1
    while (2**(k+1)) * bytes_por_amplitud <= ram:
        k += 1
    print(f"{nombre:<26}{ram/GB:>13,.0f} GB{k:>18}")

print("\nLee la última columna: multiplicar la RAM por 575,000 (de laptop a Frontier)")
print("solo te compra ~19 electrones más. ESA es la pared algorítmica.")


In [ ]:
# Clásico (2^n) vs cuántico (~n qubits) — la gráfica del PDF
ns = np.arange(1, 51)
clasico = 2.0**ns
cuantico = ns.astype(float)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.semilogy(ns, clasico, lw=2.5, color="#d62728", label="Clásico: 2ⁿ amplitudes a almacenar")
ax.semilogy(ns, cuantico, lw=2.5, color="#2ca02c", label="Cuántico: ~n qubits")
ax.axvline(20, ls="--", color="gray")
ax.annotate("n = 20 electrones\n1,048,576 amplitudes\nvs. 20 qubits",
            xy=(20, 2.0**20), xytext=(24, 1e3),
            arrowprops=dict(arrowstyle="->", color="black"), fontsize=9)
ax.set_xlabel("Número de electrones / orbitales activos (n)")
ax.set_ylabel("Recursos necesarios (escala logarítmica)")
ax.set_title("La pared algorítmica: exponencial vs. lineal")
ax.grid(alpha=0.3, which="both"); ax.legend()
plt.tight_layout(); plt.show()

print("OJO con la escala: es LOGARÍTMICA. Una recta ahí ya significa crecimiento exponencial.")
print("En escala normal la curva roja sería una pared vertical y la verde sería invisible.")


## ✅ Respuesta al Reto 2

**El cálculo pedido:** para `n = 20` electrones activos, el espacio de estados clásico es
**2²⁰ = 1,048,576 amplitudes** (≈ 16 MB en doble precisión). Con `n = 21` son 2,097,152: exactamente el doble.

**Por qué añadir un electrón duplica la memoria:** cada espín-orbital nuevo tiene dos posibilidades (ocupado o
vacío), y por la superposición hay que guardar una amplitud para **cada combinación**. Añadir un orbital significa
tomar todas las configuraciones anteriores y crear dos versiones de cada una — las que tienen el orbital lleno y
las que lo tienen vacío. De ahí el factor 2 exacto.

**Pregunta:** *¿por qué aumentar la RAM de los superordenadores no resuelve la pared algorítmica?*

**Respuesta:**

1. **Es una carrera entre crecimiento lineal y crecimiento exponencial, y la lineal siempre pierde.** Duplicar
   toda la memoria del mundo te compra **exactamente un electrón más**. Volver a duplicarla, otro. La ganancia por
   inversión es constante y ridícula frente a un problema que se duplica solo.

2. **Los números dejan de tener sentido físico enseguida.** A 300 orbitales el número de amplitudes supera el
   número estimado de átomos del universo observable (~10⁸⁰). No es un problema de presupuesto o de ingeniería:
   **no hay materia suficiente en el universo** para construir esa memoria. Ningún avance tecnológico cambia eso.

3. **El problema no es la velocidad, es la representación.** Aunque tuvieras un procesador infinitamente rápido,
   seguirías necesitando *escribir* 2ⁿ números en algún sitio. La pared es de **espacio**, no de tiempo, y por eso
   "esperar a que los chips mejoren" tampoco funciona.

4. **La ley de Moore ya se está agotando, y aun cuando no lo estaba, no bastaba.** Moore duplicaba
   aproximadamente cada dos años: al ritmo del problema, eso son **dos años de progreso de toda la industria por
   cada electrón añadido**.

5. **La solución no es tener más memoria, es no necesitarla.** Un ordenador cuántico no *almacena* las 2ⁿ
   amplitudes: las **encarna físicamente**. n espín-orbitales se mapean a n qubits — crecimiento **lineal**. Los
   estados cuánticos del registro ya son superposiciones, así que la información vive en el propio sistema físico
   en vez de en una tabla en RAM.

**En una frase:** *más RAM no arregla la pared algorítmica porque el problema no se resuelve almacenando más
rápido, sino dejando de almacenar — y eso solo lo consigue un sistema que sea él mismo cuántico.*


---
# RETO 3 — Mapeo de orbitales y propiedades de los qubits

## 🧠 Entiende esto primero

### El mapeo: 1 espín-orbital = 1 qubit
La traducción es directísima y por eso funciona tan bien:

```
qubit en |0⟩  →  orbital VACÍO
qubit en |1⟩  →  orbital OCUPADO por un electrón
```

Esto se llama **codificación por número de ocupación**. Con 4 qubits describes 4 espín-orbitales, y el estado
`|1010⟩` significa "orbitales 1 y 3 ocupados, 2 y 4 vacíos".

### El regalo: el principio de exclusión de Pauli sale gratis
El principio de Pauli dice que **dos electrones no pueden ocupar el mismo espín-orbital**. En un cálculo clásico
hay que imponer esa regla a mano, con antisimetrización y determinantes de Slater — trabajo extra y costoso.

En un qubit **no hace falta**: un qubit solo puede valer 0 o 1. **Es físicamente incapaz de representar "2
electrones en el mismo orbital"**. La restricción está incorporada en el hardware.

### Superposición: muchas configuraciones a la vez
Un bit clásico guarda **una** configuración. Un registro de `n` qubits en superposición representa **las 2ⁿ
configuraciones simultáneamente**, cada una con su amplitud. Y eso es exactamente lo que es una molécula real: no
"está" en una configuración, es una mezcla de todas.

### Entrelazamiento: la correlación, gratis
La **correlación electrónica** es que los electrones se esquivan: si uno está aquí, el otro tiende a estar allá.
Clásicamente hay que guardar una tabla gigante de esas dependencias.

El **entrelazamiento** es literalmente esa misma estructura: dos qubits entrelazados tienen resultados
correlacionados sin que ninguno tenga un valor propio definido. No modelas la correlación con una tabla: la
**reproduces físicamente**.


In [ ]:
# El mapeo en acción: 4 espín-orbitales -> 4 qubits
print("CODIFICACIÓN POR NÚMERO DE OCUPACIÓN\n")
ejemplos = {
    "0000": "Todos los orbitales vacíos (molécula ionizada)",
    "0011": "Orbitales 0 y 1 ocupados -> estado fundamental típico de H2",
    "0101": "Un electrón excitado a un orbital superior",
    "1111": "Todos los orbitales ocupados (capa llena)",
}
print(f"{'Estado':<10}{'Ocupación':<28}Significado")
print("-" * 92)
for estado, sentido in ejemplos.items():
    ocupados = [i for i, b in enumerate(reversed(estado)) if b == "1"]
    print(f"|{estado}>    {str(ocupados):<28}{sentido}")

print("\n>>> PRINCIPIO DE PAULI: fíjate que NO EXISTE un estado como |2000>.")
print("    Un qubit solo puede valer 0 o 1, así que es IMPOSIBLE poner dos electrones")
print("    en el mismo espín-orbital. La regla no se programa: viene de fábrica.")


In [ ]:
# Superposición: un registro de 3 qubits representa 8 configuraciones a la vez
n_q = 3
qc_sup = QuantumCircuit(n_q)
qc_sup.h(range(n_q))       # Hadamard en todos = superposición uniforme

estado = Statevector(qc_sup)
print(f"Con {n_q} qubits en superposición, el registro contiene {2**n_q} configuraciones A LA VEZ:\n")
for base, amp in estado.to_dict().items():
    print(f"  |{base}>  amplitud {amp.real:+.3f}   probabilidad {abs(amp)**2:.3f}")

print(f"\nUn registro CLÁSICO de {n_q} bits guardaría UNA sola de estas 8 filas.")
print(f"Con 50 orbitales: el cuántico usa 50 qubits; el clásico necesitaría")
print(f"{2**50:,} filas -> {2**50*16/1024**4:,.0f} TB. Esa es toda la diferencia.")


In [ ]:
# Entrelazamiento = correlación electrónica reproducida físicamente
qc_ent = QuantumCircuit(2)
qc_ent.h(0)
qc_ent.cx(0, 1)

estado_ent = Statevector(qc_ent)
print("Estado entrelazado (tipo Bell):")
for base, amp in estado_ent.to_dict().items():
    print(f"  |{base}>  probabilidad {abs(amp)**2:.3f}")

print("\nLéelo como química:")
print("  - Nunca sale |01> ni |10>: los dos orbitales SIEMPRE están ambos llenos o ambos vacíos.")
print("  - Ningún qubit tiene valor propio definido; solo la PAREJA está definida.")
print("  - Eso es exactamente una correlación electrónica perfecta.\n")

# Comprobación: ¿se puede describir cada qubit por separado? (si no, está entrelazado)
from qiskit.quantum_info import partial_trace, entropy
rho_q0 = partial_trace(estado_ent, [1])
print(f"Entropía del qubit 0 aislado: {entropy(rho_q0):.3f}")
print("Entropía 0 = independiente · Entropía 1 = máximamente entrelazado.")
print(">>> Sale 1: el qubit 0 NO tiene identidad propia. Toda la información está en la RELACIÓN.")
print("\nY lo importante: esto NO cuesta memoria extra. La correlación no se almacena,")
print("se REPRODUCE con una sola compuerta CNOT.")


In [ ]:
# El costo de describir lo mismo clásicamente
print(f"{'Orbitales':>10}{'Qubits necesarios':>20}{'Amplitudes clásicas':>24}{'RAM clásica':>18}")
print("-" * 74)
for n_orb in [2, 10, 20, 30, 40, 50]:
    amps = 2**n_orb
    ram = amps * 16
    if ram < 1024**3:
        ram_txt = f"{ram/1024**2:,.1f} MB"
    elif ram < 1024**4:
        ram_txt = f"{ram/1024**3:,.1f} GB"
    else:
        ram_txt = f"{ram/1024**4:,.0f} TB"
    print(f"{n_orb:>10}{n_orb:>20}{amps:>24,}{ram_txt:>18}")

print("\nColumna 2 (cuántico): crece de 1 en 1.  Columna 3 (clásico): se duplica cada fila.")
print("Misma información física, dos costos completamente distintos.")


## ✅ Respuesta al Reto 3

**Pregunta:** *¿cómo se benefician las simulaciones moleculares de la capacidad de los qubits para representar
múltiples configuraciones electrónicas simultáneamente, frente a la lógica binaria tradicional?*

**Respuesta:**

**1. La representación deja de ser una tabla y pasa a ser un estado físico.**
Un registro clásico de `n` bits guarda **una** configuración electrónica; para describir la molécula real hay que
guardar las 2ⁿ configuraciones con sus amplitudes en una tabla en RAM. Un registro de `n` qubits **está** en esas
2ⁿ configuraciones a la vez: la información no se almacena, se **encarna**. Por eso el crecimiento de recursos
pasa de **exponencial (2ⁿ) a lineal (n)**.

**2. El mapeo es natural, no una traducción forzada.**
Un espín-orbital tiene exactamente dos estados (vacío/ocupado) y un qubit también (|0⟩/|1⟩). La correspondencia es
uno a uno, sin pérdida ni codificación artificial. La molécula y el registro cuántico hablan el mismo idioma.

**3. El principio de exclusión de Pauli viene incorporado.**
Clásicamente hay que imponer la antisimetría de la función de onda a mano (determinantes de Slater), lo que
consume una parte importante del cálculo. En la codificación por ocupación, **un qubit no puede valer 2**: es
físicamente imposible representar dos electrones en el mismo espín-orbital. La restricción sale gratis del
hardware en vez de costar tiempo de cómputo.

**4. El entrelazamiento reproduce la correlación electrónica en vez de aproximarla.**
La correlación es que el comportamiento de un electrón depende del de los otros. Clásicamente eso obliga a guardar
las dependencias entre todas las configuraciones (o a aproximarlas con campo medio, perdiendo precisión). Dos
qubits entrelazados **ya tienen** esa estructura: sus resultados están correlacionados aunque ninguno tenga valor
definido por separado, y se genera con **una sola compuerta CNOT**, sin memoria adicional. Como muestra el código,
la entropía de un qubit aislado es 1: toda la información vive en la relación, que es justo lo que ocurre entre
electrones correlacionados.

**5. Paralelismo intrínseco: una operación actúa sobre todas las configuraciones.**
Al aplicar una compuerta al registro, esta actúa simultáneamente sobre las 2ⁿ componentes de la superposición.
Clásicamente habría que recorrer las 2ⁿ filas de la tabla una por una.

> ⚠️ **Matiz importante (esto lo preguntan):** el paralelismo **no** significa "leer 2ⁿ respuestas gratis". Al
> medir, la superposición colapsa y obtienes **un** resultado. La ventaja real está en que los algoritmos
> cuánticos hacen **interferir** las amplitudes para que la respuesta correcta se refuerce y las incorrectas se
> cancelen. La superposición es el medio, la interferencia es el mecanismo.

**En una frase:** *la lógica binaria tiene que describir la mecánica cuántica desde fuera, con tablas que crecen
exponencialmente; los qubits obedecen las mismas reglas que los electrones, así que representan la molécula
directamente — superposición para las configuraciones, entrelazamiento para la correlación y el principio de
Pauli de regalo.*


---
# RETO 4 — Impacto industrial: reducción de tiempos de I+D

## 🧠 Entiende esto primero

### El ciclo tradicional: prueba y error físico
Hoy, descubrir un material funciona así: se propone un candidato, se sintetiza en laboratorio, se caracteriza, se
prueba, falla, y se vuelve a empezar. Cada vuelta cuesta **semanas o meses y dinero real**. En aeroespacial,
certificar un material puede tomar **hasta 10 años**.

### El cambio: simular primero, fabricar después
La simulación predictiva le da la vuelta al embudo. En vez de fabricar 1000 candidatos y ver cuál sirve, se
**simulan** los 1000, se descartan los malos en el ordenador, y solo se fabrican los 10 más prometedores.

### La clave económica: el costo de fallar tarde
Un candidato descartado en simulación cuesta céntimos. El mismo candidato descartado tras fabricarlo cuesta miles.
Y descubierto tras certificarlo, millones. **El valor no está en simular rápido, está en fallar barato y pronto.**

### Dónde encaja lo cuántico
La simulación clásica ya hace esto (DFT), pero se equivoca justo en los materiales más interesantes: catalizadores,
baterías, sistemas con metales de transición — los **fuertemente correlacionados**. Ahí el cuántico promete un
cribado en el que **sí se puede confiar**.


In [ ]:
# Modelo del embudo de I+D: tradicional vs. con cribado por simulación
candidatos = 1000
coste_sintesis = 12_000      # USD por candidato fabricado y caracterizado
coste_simulacion = 40        # USD por candidato simulado
semanas_por_sintesis = 3
sintesis_en_paralelo = 4

# --- Escenario A: prueba y error físico puro ---
coste_A = candidatos * coste_sintesis
semanas_A = candidatos * semanas_por_sintesis / sintesis_en_paralelo

# --- Escenario B: simular 1000, fabricar solo los 10 mejores ---
finalistas = 10
coste_B = candidatos * coste_simulacion + finalistas * coste_sintesis
semanas_B = 6 + finalistas * semanas_por_sintesis / sintesis_en_paralelo   # 6 semanas de cómputo

print(f"{'':<34}{'Prueba y error':>18}{'Con simulación':>18}")
print("-" * 70)
print(f"{'Candidatos evaluados':<34}{candidatos:>18,}{candidatos:>18,}")
print(f"{'Candidatos fabricados':<34}{candidatos:>18,}{finalistas:>18,}")
print(f"{'Costo total (USD)':<34}{coste_A:>18,}{coste_B:>18,}")
print(f"{'Tiempo (semanas)':<34}{semanas_A:>18,.0f}{semanas_B:>18,.0f}")
print(f"{'Tiempo (años)':<34}{semanas_A/52:>18,.1f}{semanas_B/52:>18,.1f}")

print(f"\nAhorro de costo : {(1-coste_B/coste_A)*100:.1f}%  ({coste_A-coste_B:,} USD)")
print(f"Aceleración     : {semanas_A/semanas_B:.0f}x más rápido")
print("\nY el punto clave: se EVALUARON los mismos 1000 candidatos. No se exploró menos,")
print("se fabricó menos. El descarte ocurrió donde es barato.")


In [ ]:
# El costo de descubrir un fallo, según en qué etapa aparece
etapas = [
    ("Simulación",       40,        "Cambias un parámetro y repites"),
    ("Síntesis de lab",  12_000,    "Semanas de trabajo perdidas"),
    ("Planta piloto",    450_000,   "Escalado y equipo dedicado"),
    ("Certificación",    8_000_000, "Años de ensayos + retrasos regulatorios"),
    ("Producto en campo", 50_000_000,"Retirada, responsabilidad legal, reputación"),
]

print(f"{'Etapa donde se detecta el fallo':<28}{'Costo (USD)':>16}   Consecuencia")
print("-" * 96)
for etapa, coste, consecuencia in etapas:
    print(f"{etapa:<28}{coste:>16,}   {consecuencia}")

print(f"\nDetectarlo en simulación en vez de en campo es {50_000_000/40:,.0f}x más barato.")
print("Por eso el argumento NO es 'la simulación es rápida', sino 'fallar pronto es barato'.")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))

# Embudo
etapas_f = ["Candidatos\niniciales", "Tras cribado\nsimulado", "Síntesis", "Certificado"]
tradicional = [1000, 1000, 1000, 1]
con_sim     = [1000, 10, 10, 1]
x = np.arange(len(etapas_f)); w = 0.38
ax1.bar(x - w/2, tradicional, w, color="#d62728", label="Prueba y error")
ax1.bar(x + w/2, con_sim, w, color="#2ca02c", label="Con simulación")
ax1.set_yscale("log"); ax1.set_xticks(x); ax1.set_xticklabels(etapas_f, fontsize=8)
ax1.set_ylabel("Candidatos (escala log)"); ax1.set_title("El embudo de I+D")
ax1.legend(fontsize=8); ax1.grid(alpha=0.3, axis="y")

# Costo de fallar tarde
nombres = [e[0] for e in etapas]; costes = [e[1] for e in etapas]
ax2.barh(nombres, costes, color=plt.cm.Reds(np.linspace(0.35, 0.95, len(costes))))
ax2.set_xscale("log"); ax2.set_xlabel("Costo de detectar el fallo (USD, escala log)")
ax2.set_title("Fallar pronto es barato"); ax2.grid(alpha=0.3, axis="x")
ax2.invert_yaxis()

plt.tight_layout(); plt.show()


## ✅ Respuesta al Reto 4

**Pregunta:** *¿de qué manera la simulación cuántica de materiales puede transformar el ciclo de vida de desarrollo
de productos en industrias de alta tecnología, reduciendo la dependencia del método de prueba y error?*

**Respuesta:**

**1. Invierte el embudo: primero descartar, después fabricar.**
El ciclo tradicional fabrica para aprender. El ciclo con simulación **aprende para decidir qué fabricar**. Se
criban miles de candidatos en el ordenador y solo llegan al laboratorio los que ya superaron el filtro. Como
muestra el modelo, se evalúan los mismos 1000 candidatos pero se fabrican 10: el espacio explorado no se reduce,
se reduce el gasto físico.

**2. El valor real está en fallar pronto, no en simular rápido.**
Un error detectado en simulación cuesta decenas de dólares; el mismo error detectado en certificación cuesta
millones y años de retraso. Mover las decisiones hacia la izquierda del ciclo de vida es donde se genera casi todo
el ahorro.

**3. Convierte un proceso secuencial en uno paralelo.**
Sintetizar es intrínsecamente secuencial: hay un número limitado de bancos de laboratorio y cada prueba tarda
semanas. Simular es paralelizable: miles de candidatos a la vez. Eso es lo que comprime de **años a meses**, tal
como señala el PDF para el sector aeroespacial (hasta 10 años de certificación tradicional).

**4. Permite explorar lo que el laboratorio nunca exploraría.**
Muchos candidatos ni se prueban porque sintetizarlos es caro, peligroso o lento. En simulación no hay ese filtro
previo, así que se pueden evaluar materiales exóticos, condiciones extremas o compuestos inestables. **Se amplía
el espacio de búsqueda**, no solo se abarata.

**5. Dónde entra específicamente lo cuántico (y no basta lo clásico).**
La simulación clásica (DFT) ya hace cribado y funciona bien en química orgánica sencilla. Pero falla justo en los
materiales de mayor valor industrial — **catalizadores, baterías, captura de carbono, superconductores** — porque
son sistemas **fuertemente correlacionados**, donde el campo medio no describe bien la física. La promesa cuántica
no es "cribar más rápido", es **cribar con precisión suficiente para confiar en el resultado** justo donde hoy no
se puede.

**6. Aplicaciones concretas del PDF.**
Baterías (más densidad energética por gramo), celdas solares y semiconductores de nueva generación, catalizadores
para energía limpia, captura de carbono, y certificación acelerada de materiales aeroespaciales.

**El matiz honesto:** hoy esto es **potencial en maduración**, no práctica industrial rutinaria. El hardware NISQ
todavía no alcanza la precisión química de forma fiable (es el tema del Bloque 2). El impacto real actual es
**híbrido**: cribado clásico masivo + cálculo cuántico reservado para el núcleo fuertemente correlacionado que
decide el resultado.

**En una frase:** *la simulación transforma el I+D al mover la decisión desde el laboratorio hacia el ordenador —
donde equivocarse es barato, reversible y paralelizable — y lo cuántico extiende esa capacidad a los materiales
correlacionados donde los métodos clásicos dejan de ser confiables.*


---
# Cierre: las 5 ideas del Bloque 1

1. **El problema no es la velocidad, es el tamaño.** El espacio de estados crece como **2ⁿ**: cada electrón
   **duplica** la memoria necesaria. Eso es la **pared algorítmica**.
2. **Más RAM no la tumba.** Duplicar toda la memoria del mundo te compra **un electrón más**. A 300 orbitales
   necesitarías más memoria que átomos hay en el universo.
3. **Los qubits escalan linealmente.** 1 espín-orbital = 1 qubit. No almacenan las configuraciones: **las son**.
4. **Superposición + entrelazamiento + Pauli gratis.** Superposición = muchas configuraciones a la vez;
   entrelazamiento = correlación electrónica reproducida físicamente; y un qubit no puede valer 2, así que el
   principio de exclusión viene de fábrica.
5. **Simulador para desarrollar, QPU para validar.** El simulador da el valor exacto pero choca con la misma pared
   (~30-50 qubits); la QPU escala pero tiene ruido. Se necesitan los dos, y en ese orden.

### Preguntas de reflexión del cierre (por si las piden)
- **¿Por qué "más memoria clásica" no es sostenible?** Porque enfrenta crecimiento lineal contra crecimiento
  exponencial: cada duplicación de hardware compra un solo electrón más, y el problema pronto exige más memoria de
  la que permite la materia disponible en el universo. La salida no es almacenar mejor, es **no tener que
  almacenar**.
- **¿Qué industria impactaría más?** Baterías y energía limpia: los catalizadores y electrolitos son sistemas
  fuertemente correlacionados donde DFT es poco fiable, el valor económico de acertar es enorme, y cada iteración
  física es lenta y cara — las tres condiciones que justifican el enfoque.

---

**Siguiente sesión (Bloque 2 · Martes):** el **operador hamiltoniano**, la pieza matemática que representa la
energía del sistema, y cómo se traduce a operadores de Pauli con Qiskit Nature.
